# A05: Gestión de Memoria e Internals de CPython

---

## Objetivos de aprendizaje

1. Comprender el modelo de memoria de CPython: objetos, referencias e identidad vs igualdad.
2. Dominar el conteo de referencias (`sys.getrefcount`) y su relación con el ciclo de vida de los objetos.
3. Entender el garbage collector generacional y por qué es necesario a pesar del refcount.
4. Aplicar `__slots__` para optimizar el consumo de memoria en clases personalizadas.
5. Conocer el interning de strings, el caching de small ints y la inspección de bytecode con `dis` y `cProfile`.

## Analogía: el almacén de CPython

Imagina un **almacén gigante** donde cada objeto que creas es una caja con un contenido único.

- **Etiquetas (referencias)**: Cada variable o lista que apunta a una caja pega una etiqueta con el nombre de la caja. Si dos variables apuntan al mismo objeto, hay dos etiquetas en la misma caja.
- **Contador de etiquetas (refcount)**: En la parte trasera de cada caja hay un contador que muestra cuántas etiquetas la apuntan. Cuando el contador llega a 0, la caja se descarta.
- **Equipo de limpieza (Garbage Collector)**: A veces alguien deja cajas en un rincón apuntándose entre sí (ciclos de referencia) y el contador nunca llega a 0. El GC es el equipo de limpieza que detecta estos patrones y los recicla.
- **Etiquetas reutilizadas (interning)**: Para cajas muy comunes (números pequeños, textos cortos), el almacén reutiliza las mismas cajas en lugar de crear nuevas.

```
╔══════════════════════════════════════════════════════════════╗
║                  ALMACÉN DE CPython                         ║
║                                                             ║
║  ┌─────────┐  ┌─────────┐  ┌─────────┐  ┌─────────┐       ║
║  │  CAJA   │  │  CAJA   │  │  CAJA   │  │  CAJA   │       ║
║  │  (obj)  │  │  (obj)  │  │  (obj)  │  │  (obj)  │       ║
║  │ ref=3   │  │ ref=1   │  │ ref=0   │  │ ref=2   │       ║
║  │ ▲ ▲ ▲   │  │ ▲       │  │ LIBRE   │  │ ▲ ▲     │       ║
║  │ │ │ │   │  │ │       │  │ → FREE  │  │ │ │     │       ║
║  └─┼─┼─┼───┘  └─┼───────┘  └─────────┘  └─┼─┼─────┘       ║
║    │ │ │        │                          │ │              ║
║   a  b  c      d                            e  f            ║
║                                                             ║
║  ┌──────────────────────────────────────────────┐           ║
║  │  🧹 GC: Recorre y limpia cajas con ciclos   │           ║
║  └──────────────────────────────────────────────┘           ║
╚══════════════════════════════════════════════════════════════╝
```

---
## 1. Objetos y el modelo de memoria

En CPython **todo es un objeto**: enteros, funciones, clases, módulos, incluso `None`. Cada objeto vive en una dirección de memoria que CPython gestiona internamente.

In [ ]:
import sys

# Todo es un objeto en CPython
print(f'Tipo de 42:       {type(42)}')
print(f'Tipo de "hola":   {type("hola")}')
print(f'Tipo de None:     {type(None)}')
print(f'Tipo de una lista: {type([1, 2, 3])}')
print(f'Tipo de print:    {type(print)}')

### Referencia vs valor (aliasing)

Cuando escribes `b = a`, **no se copia** el objeto: ambas variables apuntan al **mismo** objeto en memoria.

In [ ]:
a = [1, 2, 3]
b = a  # b es un alias de a: misma dirección

print(f'id(a) = {id(a)}')
print(f'id(b) = {id(b)}')
print(f'a is b → {a is b}')  # True: misma dirección

b.append(4)  # Modificar b también modifica a
print(f'a = {a}')  # [1, 2, 3, 4]

### `id()` y su significado

En CPython, `id(obj)` retorna la **dirección de memoria** del objeto (específicamente `C:\Users\luisj\Github\Datajupyer\consolidado\avanzado\A05_internals_memoria.ipynb` en bytes). No es un identificador lógico arbitrario: es la dirección real del bloque de memoria.

In [ ]:
x = 999999
print(f'id(x)  = {id(x)}')
print(f'hex(x) = {hex(id(x))}')  # Dirección en hexadecimal

### `is` vs `==` (repaso avanzado)

| Operador | Pregunta | Implementación |
|----------|----------|----------------|
| `is` | ¿Son el **mismo objeto** en memoria? | `id(a) == id(b)` |
| `==` | ¿Tienen el **mismo valor**? | `a.__eq__(b)` |

Advertencia: `is` con literales es peligroso porque CPython optimiza (caching de small ints).

In [ ]:
# == compara valor, is compara identidad
a = [1, 2]
b = [1, 2]
print(f'a == b → {a == b}')   # True: mismo valor
print(f'a is b → {a == b}')   # False: objetos distintos
print(f'a is b → {a is b}')   # False: objetos distintos

# CUIDADO: small ints caching hace que is funcione "por accidente"
print(f'\n256 is 256 → {256 is 256}')   # True (caché en CPython)
print(f'257 is 257 → {257 is 257}')   # Puede ser True (depende del REPL)
print(f'\nNunca uses is para comparar valores, solo para None:')
print(f'x is None → {None is None}')    # La forma correcta

### Mutable vs inmutable en memoria

- **Inmutables** (`int`, `str`, `tuple`, `frozenset`): al "modificarlos" se crea un **nuevo** objeto.
- **Mutables** (`list`, `dict`, `set`, `bytearray`): se modifica **in-place** sin crear nuevo objeto.

In [ ]:
# Inmutable: cada "modificación" crea un objeto nuevo
s = "hola"
id_inicial = id(s)
s += " mundo"
id_final = id(s)
print(f'Antes: id={id_inicial}')
print(f'Después: id={id_final}')
print(f'Mismo objeto: {id_inicial == id_final}')  # False: objeto nuevo

print()

# Mutable: se modifica in-place
lst = [1, 2, 3]
id_inicial = id(lst)
lst.append(4)
id_final = id(lst)
print(f'Antes: id={id_inicial}')
print(f'Después: id={id_final}')
print(f'Mismo objeto: {id_inicial == id_final}')  # True: mismo objeto

---
## 2. Conteo de referencias

CPython usa un **contador de referencias** como mecanismo principal de gestión de memoria. Cada objeto lleva un contador interno (`ob_refcnt`) que indica cuántas referencias apuntan a él.

In [ ]:
import sys

# sys.getrefcount muestra el refcount actual
# NOTA: la propia llamada a getrefcount agrega +1 temporalmente
x = "texto_largo_para_evitar_interning"
print(f'refcount de x: {sys.getrefcount(x)}')  # 2: x + argumento de getrefcount

# Creamos un alias
y = x
print(f'refcount tras alias (y=x): {sys.getrefcount(x)}')  # 3: x + y + getrefcount

# Eliminamos un alias
del y
print(f'refcount tras del y: {sys.getrefcount(x)}')  # 2 de nuevo

### ¿Qué incrementa el refcount?

1. **Asignación directa**: `a = obj`
2. **Parámetro de función**: pasar como argumento
3. **Almacenamiento en contenedor**: agregar a una lista, diccionario, etc.
4. **Asignación a variable local**: dentro de un `try`/`except` (el traceback queda en `sys.exc_info`)

In [ ]:
obj = [1, 2, 3]
print(f'Inicial:          refcount = {sys.getrefcount(obj)}')

# Asignación a nueva variable
alias1 = obj
print(f'Alias1 = obj:     refcount = {sys.getrefcount(obj)}')

# Almacenar en un contenedor
mi_lista = [obj]
print(f'Dentro de lista:  refcount = {sys.getrefcount(obj)}')

# Eliminar referencias
del alias1
del mi_lista[0]
print(f'Eliminando refs:  refcount = {sys.getrefcount(obj)}')

del obj
# obj ya no existe aquí

### Ciclo de vida de un objeto

```
  ┌──────────────────────────────────────────────────────────────┐
  │                    CICLO DE VIDA                            │
  │                                                             │
  │  CREACIÓN           REFERENCIAS          DESTRUCCIÓN        │
  │  ─────────          ───────────          ────────────       │
  │  obj = Clase()  ──► refcount = N     ──► refcount == 0     │
  │                       │                    │                 │
  │  a = obj         ──► refcount += 1       │  Se llama a      │
  │  b = obj         ──► refcount += 1       │  __del__()       │
  │  del a           ──► refcount -= 1       │  y se libera     │
  │  del b           ──► refcount -= 1       │  la memoria      │
  │                       │                    │                 │
  │                       ▼                    ▼                 │
  │                  refcount == 0  ──► MEMORIA LIBERADA        │
  └──────────────────────────────────────────────────────────────┘
```

In [ ]:
class ObjetoRastreable:
    """Clase que notifica cuando es destruida."""
    _contador = 0

    def __init__(self, nombre: str) -> None:
        ObjetoRastreable._contador += 1
        self.nombre = nombre
        self.id = ObjetoRastreable._contador
        print(f'  [CREADO] ObjetoRastreable #{self.id} ("{self.nombre}")')

    def __del__(self) -> None:
        print(f'  [DESTRUIDO] ObjetoRastreable #{self.id} ("{self.nombre}")')


print('--- Bloque 1: creación y destrucción ---')
tmp = ObjetoRastreable('temporal')
print(f'  Refcount antes de del: {sys.getrefcount(tmp)}')
del tmp
print('  tmp eliminado, objeto destruido')

print('\n--- Bloque 2: refcount baja gradualmente ---')
obj = ObjetoRastreable('persistente')
print(f'  refcount = {sys.getrefcount(obj) - 1}')  # Restamos 1 por getrefcount
referencia = obj
print(f'  refcount = {sys.getrefcount(obj) - 1}')  # +1
del referencia
print(f'  refcount = {sys.getrefcount(obj) - 1}')  # -1
del obj

---
## 3. Garbage Collection cíclico

El refcount solo puede destruir objetos cuando su contador llega a 0. Pero **los ciclos de referencia** impiden que esto ocurra.

In [ ]:
import gc
import sys

# Demostración de un ciclo de referencia
class Nodo:
    """Nodo simple con referencia a otro nodo."""
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.siguiente: 'Nodo | None' = None
        print(f'  [CREADO] Nodo("{self.nombre}")')

    def __repr__(self) -> str:
        return f'Nodo("{self.nombre}")'

    def __del__(self) -> None:
        print(f'  [DESTRUIDO] Nodo("{self.nombre}")')


print('=== Creando ciclo A -> B -> A ===')
a = Nodo('A')
b = Nodo('B')
a.siguiente = b
b.siguiente = a  # ¡Ciclo!

print(f'\nrefcount de a (incluyendo getrefcount): {sys.getrefcount(a)}')
print(f'refcount de b (incluyendo getrefcount): {sys.getrefcount(b)}')

# Eliminamos las referencias externas
del a
del b
print('\nDel a y del b ejecutados, pero NADA fue destruido.')
print('Los objetos siguen vivos por el ciclo de referencia.')

# Forzar la recolección
print('\n=== Ejecutando gc.collect() ===')
recogidos = gc.collect()
print(f'Objetos recogidos: {recogidos}')

### ¿Por qué existe el GC si ya tenemos refcount?

1. **Ciclos de referencia**: dos o más objetos que se referencian mutuamente (refcount nunca llega a 0).
2. **Referencias en traceback**: las excepciones mantienen traceback frames que contienen referencias circulares.
3. **Referencias de extensiones C**: algunos módulos C mantienen referencias que el refcount no puede rastrear.

In [ ]:
import gc

print('=== El módulo gc: interfaz al garbage collector ===')
print()

# Estado del GC
print(f'GC habilitado:    {gc.isenabled()}')
print(f'GC debugging:     {gc.get_debug()}')  # 0 = sin debug
print()

# Umbral de generaciones (cuándo se ejecuta el GC automáticamente)
print('Umbrales de generación (gc.get_threshold()):')
print(f'  Gen 0 (frecuente): cada {gc.get_threshold()[0]} asignaciones')
print(f'  Gen 1 (media):     cada {gc.get_threshold()[1]} ciclos de gen 0')
print(f'  Gen 2 (rara):      cada {gc.get_threshold()[2]} ciclos de gen 1')
print()

# Estadísticas por generación
print('Estadísticas por generación:')
for i, stats in enumerate(gc.get_stats()):
    col = stats['collections']
    coll = stats['collected']
    uncol = stats['uncollectable']
    print(f'  Gen {i}: collections={col}, collected={coll}, uncollectable={uncol}')

In [ ]:
import gc

# gc.get_objects() retorna TODOS los objetos gestionados por el GC
todos_los_objetos = gc.get_objects()
print(f'Total de objetos rastreados por GC: {len(todos_los_objetos)}')
print()

# Contar por tipo (top 10)
from collections import Counter
tipos = Counter(type(obj).__name__ for obj in todos_los_objetos)
print('Top 10 tipos de objetos en memoria:')
for nombre, cantidad in tipos.most_common(10):
    print(f'  {nombre:<20s} → {cantidad:>8,}')

### GC Generacional: cómo funciona

CPython divide los objetos en **3 generaciones**. Los objetos más nuevos están en Gen 0 (revisados con más frecuencia) y los que sobreviven se "promueven" a generaciones superiores.

```
╔═══════════════════════════════════════════════════════════════════════╗
║                   GC GENERACIONAL DE CPython                        ║
║                                                                     ║
║  Gen 0 (NUEVOS)          Gen 1 (MEDIANOS)       Gen 2 (ANTIGUOS)   ║
║  ┌──────────────┐        ┌──────────────┐       ┌──────────────┐   ║
║  │ ████ ████    │  prom. │ ████████     │ prom. │ ████████████ │   ║
║  │ ██  ████     │ ─────► │ ████████     │─────► │ ████████████ │   ║
║  │ ████ ██      │        │ ████████     │       │ ████████████ │   ║
║  └──────────────┘        └──────────────┘       └──────────────┘   ║
║                                                                     ║
║  Frecuencia:  CADA 10        CADA 10            CADA 10             ║
║               gen0            gen0                gen0+gen1          ║
║                                                                     ║
║  Objetos nuevos ──► Gen 0 ──► Gen 1 ──► Gen 2 (permanentes)        ║
║                                                                     ║
║  Lógica: Los objetos que sobreviven más tiempo probablemente        ║
║          vivirán más, así que los revisamos con menos frecuencia.   ║
╚═══════════════════════════════════════════════════════════════════════╝
```

In [ ]:
import gc

print('=== Demostración del GC generacional ===')
print()

# Estadísticas iniciales
stats_inicial = gc.get_stats()

# Crear muchos objetos con ciclos
class Enlace:
    def __init__(self, valor: int) -> None:
        self.valor = valor
        self.enlace: 'Enlace | None' = None

print('Creando 1000 objetos con ciclos...')
for i in range(1000):
    a = Enlace(i)
    b = Enlace(i)
    a.enlace = b
    b.enlace = a  # Ciclo

# Forzar recolección completa
recolectados = gc.collect()
print(f'\nRecolectados por gc.collect(): {recolectados}')
print()

# Estadísticas después
stats_final = gc.get_stats()
for i in range(3):
    col = stats_final[i]['collections'] - stats_inicial[i]['collections']
    uncol = stats_final[i]['uncollectable'] - stats_inicial[i]['uncollectable']
    print(f'Gen {i}: +{col} collections, +{uncol} uncollectable')

---
## 4. `__slots__` y optimización de memoria

Por defecto, cada instancia de una clase tiene un `__dict__` (diccionario) que almacena sus atributos. Esto consume ~100-200 bytes por objeto. `__slots__` elimina ese diccionario.

In [ ]:
import sys


class PersonaDict:
    """Clase normal con __dict__ (por defecto)."""
    def __init__(self, nombre: str, edad: int, email: str) -> None:
        self.nombre = nombre
        self.edad = edad
        self.email = email


class PersonaSlots:
    """Clase optimizada con __slots__."""
    __slots__ = ('nombre', 'edad', 'email')

    def __init__(self, nombre: str, edad: int, email: str) -> None:
        self.nombre = nombre
        self.edad = edad
        self.email = email


# Comparar tamaño en memoria
p_dict = PersonaDict('Ana', 30, 'ana@ejemplo.com')
p_slots = PersonaSlots('Ana', 30, 'ana@ejemplo.com')

size_dict = sys.getsizeof(p_dict) + sys.getsizeof(p_dict.__dict__)
size_slots = sys.getsizeof(p_slots)

print('=== Comparación de memoria ===')
print(f'PersonaDict:  {size_dict:>5} bytes  (incluye __dict__)')
print(f'PersonaSlots: {size_slots:>5} bytes  (sin __dict__)')
print(f'Ahorro:        {size_dict - size_slots:>5} bytes por objeto')
print(f'              {(1 - size_slots / size_dict) * 100:>5.1f}% menos')
print()
print(f'PersonaDict  tiene __dict__: {hasattr(p_dict, "__dict__")}')
print(f'PersonaSlots tiene __dict__: {hasattr(p_slots, "__dict__")}')

In [ ]:
import sys

# Impacto a escala
N = 100_000

class ObjetoPesado:
    __slots__ = ()

class ObjetoLigero:
    pass

# Crear N objetos de cada tipo
import gc
gc.disable()  # Desactivar GC para medir con precisión

pesados = [ObjetoPesado() for _ in range(N)]
memoria_pesados = sum(sys.getsizeof(o) for o in pesados)
del pesados
gc.collect()

ligeros = [ObjetoLigero() for _ in range(N)]
memoria_ligeros = sum(sys.getsizeof(o) + sys.getsizeof(getattr(o, '__dict__', None)) for o in ligeros)
del ligeros
gc.enable()
gc.collect()

print(f'=== Impacto a escala ({N:,} objetos) ===')
print(f'Con __slots__:    {memoria_pesados:>12,} bytes')
print(f'Sin __slots__:    {memoria_ligeros:>12,} bytes')
print(f'Ahorro total:     {memoria_ligeros - memoria_pesados:>12,} bytes')

### ¿Cuándo usar `__slots__`?

| Situación | ¿Usar `__slots__`? |
|-----------|---------------------|
| Clases con millones de instancias | Sí |
| Datos de prueba o scripts rápidos | No importa |
| Necesitas `__dict__` dinámico | No |
| Herencia múltiple con slots conflictivos | Cuidado |
| Desarrollo rápido con atributos ad-hoc | No |

**Desventajas**:
- No puedes agregar atributos que no estén en `__slots__`.
- No hay `__dict__` por defecto (a menos que lo incluyas explícitamente).
- Puede complicar la herencia múltiple.

---
## 5. Internado de strings (interned) y caching de small ints

CPython optimiza la memoria reutilizando objetos para valores pequeños y cadenas "similares al identificador".

In [ ]:
import sys

print('=== Small Integer Caching (-5 a 256) ===')
print()

# CPython cachea enteros del -5 al 256
a = 256
b = 256
print(f'256 is 256 → {a is b}')   # True: mismo objeto caché

a = 257
b = 257
print(f'257 is 257 → {a is b}')   # Depende del contexto (REPL vs script)

a = -5
b = -5
print(f'-5  is -5  → {a is b}')   # True: dentro del rango caché

a = -6
b = -6
print(f'-6  is -6  → {a is b}')   # False: fuera del rango

print(f'\nRango de cache: -5 a 256 (inclusive)')
print(f'Documentación oficial: Objects/longobject.c en CPython')

In [ ]:
import sys

print('=== String Interning ===')
print()

# CPython interna automáticamente cadenas que parecen identificadores
# (solo letras, dígitos, guion bajo; no empiezan con número)
a = "variable_valida"
b = "variable_valida"
print(f'"variable_valida" is "variable_valida" → {a is b}')  # True: auto-interned

# Cadena con espacio: NO se interna automáticamente
a = "con espacio"
b = "con espacio"
print(f'"con espacio" is "con espacio" → {a is b}')  # Puede ser False

# sys.intern() fuerza el internado manualmente
a = sys.intern("cadena larga con espacios que quiero reutilizar")
b = sys.intern("cadena larga con espacios que quiero reutilizar")
print(f'\nsys.intern() fuerza internado → {a is b}')  # True
print(f'Misma dirección: {id(a) == id(b)}')

In [ ]:
import sys

# Impacto real del interning: medir memoria
texto_largo = "x" * 1000  # Sin interning
texto_interned = sys.intern("x" * 1000)  # Con interning

# Crear muchas copias
lista_normal = ["x" * 1000 for _ in range(1000)]
lista_interned = [sys.intern("x" * 1000) for _ in range(1000)]

memoria_normal = sum(sys.getsizeof(s) for s in lista_normal)
memoria_interned = sum(sys.getsizeof(s) for s in lista_interned)

print('=== Impacto del interning a escala ===')
print(f'Sin intern: {memoria_normal:>10,} bytes ({len(lista_normal)} cadenas)')
print(f'Con intern: {memoria_interned:>10,} bytes ({len(lista_interned)} cadenas)')
print(f'Ahorro:     {memoria_normal - memoria_interned:>10,} bytes')
print(f'Objetos únicos (sin intern): {len(set(id(s) for s in lista_normal))}')
print(f'Objetos únicos (con intern): {len(set(id(s) for s in lista_interned))}')

---
## 6. Inspección de bytecode con `dis`

El módulo `dis` (disassembler) decompila funciones Python a su **bytecode** equivalente. Esto revela cómo CPython ejecuta tu código a nivel de máquina virtual.

In [ ]:
import dis

# Función simple para inspeccionar
def suma_rapida(a: int, b: int) -> int:
    x = a + b
    return x

print('=== Bytecode de suma_rapida ===')
print()
dis.dis(suma_rapida)

In [ ]:
import dis

# Comparar LOAD_FAST vs LOAD_GLOBAL
resultado_global = 0

def usa_global() -> int:
    """Usa una variable global: más lenta."""
    global resultado_global
    for i in range(100):
        resultado_global += i
    return resultado_global


def usa_local() -> int:
    """Usa una variable local: más rápida."""
    resultado = 0
    for i in range(100):
        resultado += i
    return resultado


print('=== Bytecode de usa_global ===')
print()
dis.dis(usa_global)

print('\n=== Bytecode de usa_local ===')
print()
dis.dis(usa_local)

print('\nObservación: LOAD_GLOBAL es más costoso que LOAD_FAST.')
print('Las variables locales se almacenan en un array (más rápido).')

In [ ]:
import dis
import timeit

# Medir la diferencia de velocidad real
setup = '''
var_global = 0
'''

stmt_global = '''
global var_global
for i in range(1000):
    var_global += i
'''

stmt_local = '''
resultado = 0
for i in range(1000):
    resultado += i
'''

t_global = timeit.timeit(stmt_global, setup=setup, number=10000)
t_local = timeit.timeit(stmt_local, number=10000)

print('=== Benchmark: Global vs Local ===')
print(f'Variable global: {t_global:.4f}s')
print(f'Variable local:  {t_local:.4f}s')
print(f'Ratio:           {t_global / t_local:.2f}x (local es {t_global/t_local:.1f}x más rápido)')

### Instrucciones de bytecode más comunes

| Instrucción | Descripción | Velocidad relativa |
|------------|-------------|--------------------|
| `LOAD_FAST` | Carga variable local (array) | Rápida |
| `LOAD_GLOBAL` | Carga variable global/módulo | Media |
| `LOAD_ATTR` | Carga atributo de objeto | Lenta |
| `LOAD_CONST` | Carga constante literal | Rápida |
| `CALL_FUNCTION` | Llama a función | Lenta |
| `BINARY_OP` | Operación aritmética | Media |

---
## 7. Profiling

Profiling = medir **dónde** se consume tiempo (y memoria) en tu programa para encontrar cuellos de botella.

In [ ]:
import cProfile
import pstats
import io


# Funciones de ejemplo con diferentes costos
def funcion_lenta() -> list[int]:
    """Simula trabajo pesado."""
    resultado = []
    for i in range(100_000):
        resultado.append(i * 2)
    return resultado


def funcion_media() -> int:
    """Simula trabajo medio."""
    return sum(range(50_000))


def funcion_rapida() -> float:
    """Simula trabajo ligero."""
    return sum(1.0 / i for i in range(1, 1000))


def programa_principal() -> None:
    """Punto de entrada del programa."""
    for _ in range(3):
        funcion_lenta()
    for _ in range(5):
        funcion_media()
    for _ in range(10):
        funcion_rapida()


# Profilear el programa
print('=== cProfile: Profile del programa ===')
print()

profiler = cProfile.Profile()
profiler.enable()
programa_principal()
profiler.disable()

# Mostrar estadísticas ordenadas por tiempo acumulado
stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream)
stats.sort_stats('cumulative')
stats.print_stats(10)
print(stream.getvalue())

In [ ]:
import cProfile
import pstats
import io

# Reutilizar las funciones del ejemplo anterior
def funcion_lenta() -> list[int]:
    return [i * 2 for i in range(100_000)]

def funcion_media() -> int:
    return sum(range(50_000))

def funcion_rapida() -> float:
    return sum(1.0 / i for i in range(1, 1000))

def programa_principal() -> None:
    for _ in range(3):
        funcion_lenta()
    for _ in range(5):
        funcion_media()
    for _ in range(10):
        funcion_rapida()


# Estadísticas por tiempo propio (tottime)
print('=== Top funciones por tiempo propio (tottime) ===')
print()

profiler = cProfile.Profile()
profiler.enable()
programa_principal()
profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream)
stats.sort_stats('tottime')
stats.print_stats(8)
print(stream.getvalue())

In [ ]:
import timeit

print('=== timeit: comparar implementaciones ===')
print()

# Comparar diferentes formas de concatenar strings
setup_concat = 'palabras = ["palabra" + str(i) for i in range(1000)]'

metodo_concat = ' resultado = ""; [resultado := resultado + p for p in palabras]'
metodo_join = 'resultado = " ".join(palabras)'

t_concat = timeit.timeit(metodo_concat, setup=setup_concat, number=1000)
t_join = timeit.timeit(metodo_join, setup=setup_concat, number=1000)

print(f'Concatenación manual: {t_concat:.4f}s')
print(f'" ".join():          {t_join:.4f}s')
print(f'Ratio:                {t_concat / t_join:.1f}x (join es más rápido)')

print()

# Comparar list comprehension vs loop explícito
t_comp = timeit.timeit('[i**2 for i in range(10000)]', number=1000)

setup_loop = '''
def cuadrados():
    resultado = []
    for i in range(10000):
        resultado.append(i**2)
    return resultado
'''
t_loop = timeit.timeit('cuadrados()', setup=setup_loop, number=1000)

print(f'List comprehension: {t_comp:.4f}s')
print(f'Loop + append:      {t_loop:.4f}s')
print(f'Ratio:              {t_loop / t_comp:.2f}x (comprehension es más rápida)')

In [ ]:
import sys
import time

print('=== memory_profiler (referencia externa) ===')
print()
print('memory_profiler NO está en stdlib. Para instalarlo:')
print('  pip install memory_profiler')
print()
print('Uso típico con @profile decorator:')
print()
print('  from memory_profiler import profile')
print()
print('  @profile')
print('  def mi_funcion():')
print('      a = [1] * (10 ** 6)    # 8.0 MB')
print('      b = [2] * (10 ** 7)    # 80.0 MB')
print('      del b                   # 0.0 MB')
print('      return a')
print()
print('Salida esperada:')
print('  Line #    Mem usage    Increment   Line Contents')
print('  ===============================================')
print('       3     35.0 MiB      0.0 MiB   @profile')
print('       4     42.8 MiB      7.8 MiB   a = [1] * (10 ** 6)')
print('       5    119.3 MiB     76.5 MiB   b = [2] * (10 ** 7)')
print('       6     42.8 MiB    -76.5 MiB   del b')
print()
print('Alternativas sin instalar:')
print('  - sys.getsizeof(obj) → tamaño de un solo objeto')
print('  - Tracemalloc (stdlib) → rastrea asignaciones de memoria')

print()
print('=== Uso de tracemalloc (stdlib) ===')
import tracemalloc

tracemalloc.start()

# Crear algunos objetos
datos_grandes = [bytearray(10000) for _ in range(1000)]

snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics('lineno')

print('Top 5 líneas por uso de memoria:')
for stat in top_stats[:5]:
    print(f'  {stat}')

tracemalloc.stop()

---
## 8. Buenas prácticas de memoria

Patrones concretos para escribir código más eficiente en memoria.

In [ ]:
import sys
import timeit

print('=== Generadores vs Listas ===')
print()

# Lista: crea todo en memoria
def cuadrados_lista(n: int) -> list[int]:
    return [i**2 for i in range(n)]

# Generador: produce valores uno a uno
def cuadrados_generador(n: int):
    return (i**2 for i in range(n))

N = 1_000_000

lista = cuadrados_lista(N)
print(f'Lista ({N:,} elementos):     {sys.getsizeof(lista):>10,} bytes')

generador = cuadrados_generador(N)
print(f'Generador (lazy):           {sys.getsizeof(generador):>10,} bytes')
print(f'Ahorro:                     {sys.getsizeof(lista) - sys.getsizeof(generador):>10,} bytes')

print()

# Cuando puedes iterar una vez, usa generadores
t_lista = timeit.timeit(
    'sum([i**2 for i in range(100000)])',
    number=100
)
t_generador = timeit.timeit(
    'sum(i**2 for i in range(100000))',
    number=100
)
print(f'Sum con lista:      {t_lista:.4f}s')
print(f'Sum con generador:  {t_generador:.4f}s')

In [ ]:
import sys

print('=== sys.getsizeof: inspeccionar tamaño ===')
print()

# Comparar tamaños de tipos comunes
elementos = [
    ('None', None),
    ('bool True', True),
    ('int 0', 0),
    ('int 1000', 1000),
    ('float 1.0', 1.0),
    ('str vacío', ''),
    ('str "hola"', 'hola'),
    ('str "hola" * 100', 'hola' * 100),
    ('tuple ()', ()),
    ('tuple (1,2,3)', (1, 2, 3)),
    ('list []', []),
    ('list [1,2,3]', [1, 2, 3]),
    ('dict {}', {}),
    ('set()', set()),
    ('bytes(0)', b''),
    ('bytes(100)', b'x' * 100),
]

print(f'{"Tipo":<20s} {"Ejemplo":<16s} {"Bytes":>8s}')
print('-' * 48)
for nombre, obj in elementos:
    print(f'{type(obj).__name__:<20s} {nombre:<16s} {sys.getsizeof(obj):>8,}')

In [ ]:
import sys
import timeit

print('=== Patrón: batch vs individual ===')
print()

# MAL: crear objetos individuales innecesariamente
def mala_practica() -> list[list[int]]:
    resultado = []
    for i in range(1000):
        sublista = []  # Nuevo objeto list cada vez
        for j in range(10):
            sublista.append(i * j)
        resultado.append(sublista)
    return resultado

# BIEN: list comprehension (más eficiente)
def buena_practica() -> list[list[int]]:
    return [[i * j for j in range(10)] for i in range(1000)]

# MEJOR: si solo necesitas iterar, usa un generador anidado
def mejor_practica():
    return ((i * j for j in range(10)) for i in range(1000))


t_mala = timeit.timeit(mala_practica, number=1000)
t_buena = timeit.timeit(buena_practica, number=1000)
t_mejor = timeit.timeit(lambda: list(mejor_practica()), number=1000)

print(f'Mal: loop + append:     {t_mala:.4f}s')
print(f'Bien: comprehension:    {t_buena:.4f}s')
print(f'Mejor: gen anidado:     {t_mejor:.4f}s')
print(f'
Ahorro de comprehension vs loop: {(1 - t_buena/t_mala) * 100:.1f}%')

---
## Diagrama ASCII: Objeto con refcount y generaciones GC

### Estructura interna de un objeto Python

```
╔════════════════════════════════════════════════════════════════╗
║         ESTRUCTURA INTERNA DE UN OBJETO PYTHON               ║
║         (simplificado desde CPython 3.12)                     ║
╠════════════════════════════════════════════════════════════════╣
║                                                              ║
║  PyObject (base de todo)                                     ║
║  ┌─────────────────────────────────────────────┐             ║
║  │  ob_refcnt      │ 4 bytes  │ Contador de   │             ║
║  │  (refcount)     │          │ referencias   │             ║
║  ├─────────────────────────────────────────────┤             ║
║  │  ob_type        │ 8 bytes  │ Puntero al    │             ║
║  │  (tipo)         │ (ptr)    │ type object   │             ║
║  └─────────────────────────────────────────────┘             ║
║                                                              ║
║  Para un int (ob替val):                                     ║
║  ┌─────────────────────────────────────────────┐             ║
║  │  ob_ival        │ 8 bytes  │ Valor entero  │             ║
║  └─────────────────────────────────────────────┘             ║
║                                                              ║
║  Para una instancia con __dict__:                            ║
║  ┌─────────────────────────────────────────────┐             ║
║  │  PyObject base  │ 12 bytes │ refcnt+tipo   │             ║
║  ├─────────────────────────────────────────────┤             ║
║  │  __dict__       │ 8 bytes  │ ptr a dict    │             ║
║  ├─────────────────────────────────────────────┤             ║
║  │  atributos...   │ N bytes  │ datos inst.   │             ║
║  └─────────────────────────────────────────────┘             ║
║                                                              ║
║  Para una instancia CON __slots__:                           ║
║  ┌─────────────────────────────────────────────┐             ║
║  │  PyObject base  │ 12 bytes │ refcnt+tipo   │             ║
║  ├─────────────────────────────────────────────┤             ║
║  │  slot_0         │ 8 bytes  │ atributo 1    │             ║
║  ├─────────────────────────────────────────────┤             ║
║  │  slot_1         │ 8 bytes  │ atributo 2    │             ║
║  └─────────────────────────────────────────────┘             ║
║  (sin __dict__: ahorro de ~100-200 bytes por instancia)     ║
╚════════════════════════════════════════════════════════════════╝
```

### GC Generacional: proceso de recolección

```
╔════════════════════════════════════════════════════════════════╗
║              PROCESO DE RECOLECCIÓN DEL GC                    ║
╠════════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. DETECCIÓN (cada N asignaciones para Gen 0)              ║
║     ┌──────────────────────────────────────────────┐         ║
║     │  gc.collect() recorre todos los objetos      │         ║
║     │  marcando los alcanzables desde las raíces   │         ║
║     └──────────────────────────────────────────────┘         ║
║                         │                                    ║
║                         ▼                                    ║
║  2. MARCADO (mark)                                           ║
║     ┌──────────────────────────────────────────────┐         ║
║     │  Raíces: módulos, frames activos,           │         ║
║     │  globals(), repositorio de gc               │         ║
║     │                                             │         ║
║     │  ● ──► ● ──► ●   (alcanzable: sobrevive)   │         ║
║     │  ○     ○          (no alcanzable: recolecta)│         ║
║     │    ╲   ╱                                   │         ║
║     │     ╲ ╱    ciclo no alcanzable              │         ║
║     └──────────────────────────────────────────────┘         ║
║                         │                                    ║
║                         ▼                                    ║
║  3. SWEEP (sweep)                                            ║
║     ┌──────────────────────────────────────────────┐         ║
║     │  Se eliminan los objetos NO marcados         │         ║
║     │  Se promueven los sobrevivientes:            │         ║
║     │    Gen 0 → Gen 1   (si sobrevive 2 veces)   │         ║
║     │    Gen 1 → Gen 2   (si sobrevive 10 veces)  │         ║
║     └──────────────────────────────────────────────┘         ║
╚════════════════════════════════════════════════════════════════╝
```

---
## Tabla de referencia: módulos y funciones clave

| Módulo | Función / Método | Descripción | Ejemplo de uso |
|--------|-----------------|-------------|----------------|
| `sys` | `getrefcount(obj)` | Retorna el refcount de un objeto | `sys.getrefcount(x)` |
| `sys` | `getsizeof(obj)` | Tamaño en bytes del objeto (no recursivo) | `sys.getsizeof([1,2,3])` |
| `sys` | `intern(s)` | Fuerza el internado de una cadena | `s = sys.intern(cadena)` |
| `gc` | `isenabled()` | ¿GC activado? | `gc.isenabled()` |
| `gc` | `get_threshold()` | Umbrales de generaciones (gen0, gen1, gen2) | `gc.get_threshold()` |
| `gc` | `get_stats()` | Estadísticas por generación | `gc.get_stats()` |
| `gc` | `get_objects()` | Lista de todos los objetos rastreados | `gc.get_objects()` |
| `gc` | `collect()` | Forzar recolección completa | `gc.collect()` |
| `gc` | `disable()` / `enable()` | Activar/desactivar GC | `gc.disable()` |
| `dis` | `dis(obj)` | Desensambla bytecode de función/código | `dis.dis(mi_funcion)` |
| `dis` | `get_instructions(obj)` | Iterador de instrucciones de bytecode | `list(dis.get_instructions(mi_funcion))` |
| `cProfile` | `Profile()` | Profileador de tiempo por función | Ver ejemplo arriba |
| `pstats` | `Stats(profile)` | Estadísticas formateadas del profileador | `stats.print_stats(10)` |
| `timeit` | `timeit(stmt, ...)` | Mide tiempo de ejecución preciso | `timeit.timeit('sum(range(1000))')` |
| `tracemalloc` | `start()` / `take_snapshot()` | Rastrea asignaciones de memoria | `tracemalloc.start()` |
| `tracemalloc` | `take_snapshot()` | Captura estado de memoria actual | `snap = tracemalloc.take_snapshot()` |

---
## Ejercicios

### Ejercicio 1 (guiado): Explorar refcount y ciclos

**Objetivo**: Observar cómo cambia el refcount al crear y eliminar referencias, y cómo un ciclo de referencia evita la liberación.

**Instrucciones**:
1. Crea un objeto y observa su refcount con `sys.getrefcount`.
2. Crea 3 aliases y observa cómo sube el refcount.
3. Elimina los aliases y observa cómo baja.
4. Crea un ciclo entre dos objetos y elimina las referencias externas.
5. Verifica que los objetos siguen vivos y que `gc.collect()` los libera.

In [ ]:
import sys
import gc


class Nodo:
    """Nodo simple para el ejercicio."""
    _contador = 0

    def __init__(self, nombre: str) -> None:
        Nodo._contador += 1
        self.nombre = nombre
        self.id = Nodo._contador
        self.enlace: 'Nodo | None' = None

    def __repr__(self) -> str:
        return f'Nodo("{self.nombre}")'

    def __del__(self) -> None:
        print(f'  [DESTRUIDO] {self}')


# Paso 1: Crear objeto y ver refcount
print('=== Paso 1: Objeto nuevo ===')
a = Nodo('A')
print(f'  refcount de a: {sys.getrefcount(a) - 1}')  # -1 por el argumento de getrefcount

# Paso 2: Crear aliases
print('\n=== Paso 2: Creando aliases ===')
alias1 = a
alias2 = a
alias3 = a
print(f'  refcount de a: {sys.getrefcount(a) - 1}')

# Paso 3: Eliminar aliases
print('\n=== Paso 3: Eliminando aliases ===')
del alias1
del alias2
del alias3
print(f'  refcount de a: {sys.getrefcount(a) - 1}')

# Paso 4: Crear ciclo
print('\n=== Paso 4: Creando ciclo ===')
b = Nodo('B')
a.enlace = b
b.enlace = a  # ¡Ciclo!

print(f'  refcount de a: {sys.getrefcount(a) - 1}')
print(f'  refcount de b: {sys.getrefcount(b) - 1}')

print('\n=== Eliminando referencias externas ===')
del a
del b
print('  a y b eliminados, pero objetos NO destruidos (ciclo)')

# Paso 5: GC libera los ciclos
print('\n=== Paso 5: gc.collect() ===')
recogidos = gc.collect()
print(f'  Objetos recogidos: {recogidos}')
print('  Ciclo roto, objetos destruidos')

### Ejercicio 2 (guiado): Medir el impacto de `__slots__`

**Objetivo**: Comparar el consumo de memoria entre una clase normal y una clase con `__slots__`.

**Instrucciones**:
1. Define dos clases: `ParticleDict` (sin slots) y `ParticleSlots` (con `__slots__`).
2. Crea 100,000 instancias de cada una.
3. Mide el uso total de memoria con `sys.getsizeof`.
4. Reporta el ahorro en bytes y porcentaje.

In [ ]:
import sys
import gc


class ParticleDict:
    """Partícula sin __slots__ (usa __dict__)."""
    def __init__(self, x: float, y: float, z: float, name: str) -> None:
        self.x = x
        self.y = y
        self.z = z
        self.name = name


class ParticleSlots:
    """Partícula con __slots__ (sin __dict__)."""
    __slots__ = ('x', 'y', 'z', 'name')

    def __init__(self, x: float, y: float, z: float, name: str) -> None:
        self.x = x
        self.y = y
        self.z = z
        self.name = name


N = 100_000
gc.disable()

# Medir ParticleDict
dict_instances = [ParticleDict(i * 0.1, i * 0.2, i * 0.3, f'p{i}') for i in range(N)]
mem_dict = sum(sys.getsizeof(o) + sys.getsizeof(getattr(o, '__dict__', None)) for o in dict_instances)
del dict_instances
gc.collect()

# Medir ParticleSlots
slot_instances = [ParticleSlots(i * 0.1, i * 0.2, i * 0.3, f'p{i}') for i in range(N)]
mem_slots = sum(sys.getsizeof(o) for o in slot_instances)
del slot_instances
gc.enable()
gc.collect()

print(f'=== Resultados ({N:,} instancias) ===')
print(f'ParticleDict:   {mem_dict:>12,} bytes')
print(f'ParticleSlots:  {mem_slots:>12,} bytes')
print(f'Ahorro:          {mem_dict - mem_slots:>12,} bytes')
print(f'Porcentaje:     {(1 - mem_slots / mem_dict) * 100:>11.1f}%')

### Ejercicio 3 (guiado): Desensamblar y comparar funciones

**Objetivo**: Usar `dis` para entender por qué las variables locales son más rápidas.

**Instrucciones**:
1. Crea una función que use `global` y otra que use solo variables locales.
2. Desensambla ambas con `dis.dis()`.
3. Identifica las instrucciones `LOAD_GLOBAL` vs `LOAD_FAST`.
4. Mide el tiempo con `timeit` para confirmar la diferencia.

In [ ]:
import dis
import timeit


_contador_global = 0


def con_global() -> int:
    """Función que usa variable global."""
    global _contador_global
    total = 0
    for i in range(10_000):
        _contador_global += i
        total += i
    return total


def sin_global() -> int:
    """Función que solo usa variables locales."""
    total = 0
    for i in range(10_000):
        total += i
    return total


print('=== Desensamblaje: con_global ===')
dis.dis(con_global)

print('\n=== Desensamblaje: sin_global ===')
dis.dis(sin_global)

print('\n=== Benchmark ===')
t_global = timeit.timeit(con_global, number=1000)
t_local = timeit.timeit(sin_global, number=1000)
print(f'con_global:  {t_global:.4f}s')
print(f'sin_global:  {t_local:.4f}s')
print(f'Diferencia:  {t_global / t_local:.2f}x')

### Ejercicio 4 (independiente): Detector de ciclos de referencia

**Objetivo**: Crear una función que detecte y reporte ciclos de referencia en un grafo de objetos.

**Instrucciones**:
1. Crea una clase `NodoGrafo` con atributo `hijos: list[NodoGrafo]`.
2. Construye un grafo con al menos 2 ciclos.
3. Implementa una función `detectar_ciclos(raiz)` que:
   - Use DFS (depth-first search) con un conjunto de visitados.
   - Detecte cuándo se visita un nodo que ya está en la pila de recursión.
   - Retorne una lista de tuplas `(nodo_origen, nodo_destino)` que forman ciclos.
4. Valida tu detector usando `gc.get_objects()` para verificar que los objetos siguen vivos.
5. Mide el consumo de memoria de tu grafo con `sys.getsizeof` + tamaño recursivo.

In [ ]:
import sys
import gc
from typing import Optional


class NodoGrafo:
    """Nodo de un grafo con hijos."""
    _contador = 0

    def __init__(self, nombre: str) -> None:
        NodoGrafo._contador += 1
        self.nombre = nombre
        self.id = NodoGrafo._contador
        self.hijos: list[NodoGrafo] = []

    def __repr__(self) -> str:
        return f'NodoGrafo("{self.nombre}", id={self.id})'

    def agregar_hijo(self, hijo: 'NodoGrafo') -> None:
        self.hijos.append(hijo)


def detectar_ciclos(raiz: NodoGrafo) -> list[tuple[NodoGrafo, NodoGrafo]]:
    """Detecta ciclos usando DFS. Retorna lista de (origen, destino) de ciclos."""
    ciclos: list[tuple[NodoGrafo, NodoGrafo]] = []
    visitados: set[int] = set()
    pila: set[int] = set()

    def dfs(nodo: NodoGrafo) -> None:
        visitados.add(nodo.id)
        pila.add(nodo.id)

        for hijo in nodo.hijos:
            if hijo.id in pila:
                ciclos.append((nodo, hijo))
            elif hijo.id not in visitados:
                dfs(hijo)

        pila.remove(nodo.id)

    dfs(raiz)
    return ciclos


# Construir grafo con ciclos
a = NodoGrafo('A')
b = NodoGrafo('B')
c = NodoGrafo('C')
d = NodoGrafo('D')
e = NodoGrafo('E')

# Grafo: A -> B -> C -> A (ciclo 1)
#         A -> D -> E -> D (ciclo 2)
a.agregar_hijo(b)
b.agregar_hijo(c)
c.agregar_hijo(a)  # Ciclo: C -> A
a.agregar_hijo(d)
d.agregar_hijo(e)
e.agregar_hijo(d)  # Ciclo: E -> D

print('=== Grafo construido ===')
print(f'Nodo A: {[h.nombre for h in a.hijos]}')
print(f'Nodo B: {[h.nombre for h in b.hijos]}')
print(f'Nodo C: {[h.nombre for h in c.hijos]}')
print(f'Nodo D: {[h.nombre for h in d.hijos]}')
print(f'Nodo E: {[h.nombre for h in e.hijos]}')

# Detectar ciclos
ciclos_encontrados = detectar_ciclos(a)
print(f'\n=== Ciclos detectados: {len(ciclos_encontrados)} ===')
for origen, destino in ciclos_encontrados:
    print(f'  {origen} -> {destino}')

# Verificar que los objetos siguen vivos
print(f'\n=== Verificación post-ciclo ===')
print(f'Nodo A sigue vivo: {a is not None}')
print(f'Nodo B sigue vivo: {b is not None}')

# Limpiar
del a, b, c, d, e
recogidos = gc.collect()
print(f'gc.collect() liberó: {recogidos} objetos')

# Medir memoria
print(f'\n=== Memoria de NodoGrafo ===')
print(f'NodoGrafo base: {sys.getsizeof(NodoGrafo("test"))} bytes')
print(f'Nota: el dict de la instancia agrega ~100-200 bytes más')

---
## Resumen

| Concepto | Punto clave |
|----------|------------|
| **Todo es objeto** | Cada valor en Python tiene un type object y una dirección de memoria (`id()`) |
| **is vs ==** | `is` compara identidad (misma dirección), `==` compara valor (`__eq__`) |
| **Refcount** | Cada objeto lleva un contador de referencias; cuando llega a 0, se libera |
| **Ciclos de referencia** | Objetos que se referencian mutuamente impiden que refcount llegue a 0 |
| **GC generacional** | 3 generaciones (0/1/2) con frecuencias decrecientes de revisión |
| **gc.collect()** | Fuerza una recolección completa; útil para debugging |
| **`__slots__`** | Elimina `__dict__` por instancia → ahorra ~100-200 bytes/objeto |
| **Small int cache** | CPython cachea enteros del -5 al 256 (mismo objeto) |
| **String interning** | Cadenas tipo-identificador se auto-internan; `sys.intern()` fuerza más |
| **`dis`** | Desensambla bytecode; `LOAD_FAST` (local) es más rápido que `LOAD_GLOBAL` |
| **cProfile** | Profilea tiempo por función; ordena por `cumulative` o `tottime` |
| **timeit** | Mide tiempo de snippets pequeños con alta precisión |
| **tracemalloc** | Rastrea asignaciones de memoria (stdlib, alternativa a memory_profiler) |
| **Generadores** | Producen valores lazy; mucho menos memoria que listas para iteración única |
| **`sys.getsizeof`** | Mide tamaño de un objeto (no recursivo para contenedores) |

---

**Próximo tema**: `A06_internals_concurrencia` - GIL, subinterpreters y concurrencia en CPython.